In [4]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE  
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

# get the data
data = yf.download('SPY', start='2015-01-01', end='2025-1-1')

# get the features
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data.ta.ema(length=100, append=True)  # Exponential Moving Average
data.ta.sma(length=25, append=True)   # Simple Moving Average
data.ta.rsi(length=10, append=True)  # Relative Strength Index
data.ta.vwap(high='High', low='Low', close='Close', volume='Volume', append=True)  # Volume Weighted Average Price
data.ta.bbands(append=True)  # Bollinger Bands
data.ta.atr(length=14, append=True)  # Average True Range

print("--- Actual Column Names ---")
print(data.columns)
print("----------------------------")

# create label
future_return = data['Close'].pct_change(5).shift(-5)
UPPER_THRESHOLD = 0.02
LOWER_THRESHOLD = -0.02
conditions = [(future_return > UPPER_THRESHOLD), (future_return < LOWER_THRESHOLD)]
choices = [1, -1]
data['Target'] = np.select(conditions, choices, default=0)

## ------------------------------------------------------------------
## STEP 4: Prepare Data for the Model (Same)
## ------------------------------------------------------------------
data = data.dropna()
feature_list = ['EMA_100', 'SMA_25', 'RSI_10', 'VWAP_D', 'BBL_5_2.0_2.0', 'BBB_5_2.0_2.0', 'BBU_5_2.0_2.0','BBP_5_2.0_2.0',
                'ATRr_14']
X = data[feature_list]
Y = data['Target']

split_percentage = 0.8
split_index = int(len(data) * split_percentage)
X_train = X[:split_index]
Y_train = Y[:split_index]
X_test = X[split_index:]
Y_test = Y[split_index:]

print("--- Class Distribution ---")
print(Y_train.value_counts(normalize=True))


# k_neighbors must be less than the smallest class count.
# Your smallest class (Sell) has ~12.6% of ~2000 samples = ~252.
# So k_neighbors=5 is perfectly fine.
smote = SMOTE(random_state=42) 

# Fit and apply SMOTE ONLY to the training data
X_train_resampled, Y_train_resampled = smote.fit_resample(X_train, Y_train)

print("--- Class Distribution (After SMOTE) ---")
print(Y_train_resampled.value_counts(normalize=True))

## ------------------------------------------------------------------
## STEP 5: *** NEW - RUN RFE TO FIND THE BEST 2 FEATURES ***
## ------------------------------------------------------------------
# use RandomForest as the base estimator
estimator = RandomForestClassifier(n_estimators=50, random_state=42)

# use RFE to select the top 2 features
rfe = RFE(estimator=estimator, n_features_to_select=5, step=1)

# Run the RFE process on our training data
rfe.fit(X_train_resampled, Y_train_resampled)

# Print the results
print("\n--- RFE Results ---")
print(f"Original features: {feature_list}")
print(f"Selected mask:     {rfe.support_}")
print(f"Feature ranking:   {rfe.ranking_}")

# Get the names of the winning features
selected_features = X_train_resampled.columns[rfe.support_].tolist()
print(f"Selected features: {selected_features}")

rfe.fit(X_train_resampled, Y_train_resampled)

## ------------------------------------------------------------------
## STEP 6: *** NEW - PREPARE DATA WITH *ONLY* THE BEST FEATURES ***
## ------------------------------------------------------------------
# Now we filter our X data to ONLY use the features RFE selected
X_train_selected = X_train_resampled[selected_features] # <-- Use resampled
X_test_selected = X_test[selected_features] # <-- Test set stays original

print(f"\nTraining final model with {len(selected_features)} selected features...")


## ------------------------------------------------------------------
## STEP 7: Train and Evaluate the FINAL Model
## ------------------------------------------------------------------
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10]
}

# Create the grid search object
# cv=3 means it will use 3-fold cross-validation
# n_jobs=-1 means it will use all your CPU cores
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro', # Use f1-score, not accuracy!
    n_jobs=-1
)

# Fit the grid search on the *selected features* training data
print("Running GridSearchCV...")
grid_search.fit(X_train_selected, Y_train_resampled)

# See the best settings it found
print(f"Best parameters found: {grid_search.best_params_}")

# The grid_search object is now your best model
predictions = grid_search.predict(X_test_selected)

# Now check the report
print(classification_report(Y_test, predictions, target_names=['Sell (-1)', 'Hold (0)', 'Buy (1)']))

/tmp/ipykernel_778/1367659664.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('SPY', start='2015-01-01', end='2025-1-1')
[*********************100%***********************]  1 of 1 completed


--- Actual Column Names ---
Index(['Close', 'High', 'Low', 'Open', 'Volume', 'EMA_100', 'SMA_25', 'RSI_10',
       'VWAP_D', 'BBL_5_2.0_2.0', 'BBM_5_2.0_2.0', 'BBU_5_2.0_2.0',
       'BBB_5_2.0_2.0', 'BBP_5_2.0_2.0', 'ATRr_14'],
      dtype='object', name='Price')
----------------------------
--- Class Distribution ---
Target
 0    0.718572
 1    0.154682
-1    0.126746
Name: proportion, dtype: float64
--- Class Distribution (After SMOTE) ---
Target
 0    0.333333
-1    0.333333
 1    0.333333
Name: proportion, dtype: float64

--- RFE Results ---
Original features: ['EMA_100', 'SMA_25', 'RSI_10', 'VWAP_D', 'BBL_5_2.0_2.0', 'BBB_5_2.0_2.0', 'BBU_5_2.0_2.0', 'BBP_5_2.0_2.0', 'ATRr_14']
Selected mask:     [ True  True  True  True False False False False  True]
Feature ranking:   [1 1 1 1 3 4 2 5 1]
Selected features: ['EMA_100', 'SMA_25', 'RSI_10', 'VWAP_D', 'ATRr_14']

Training final model with 5 selected features...
Running GridSearchCV...
Best parameters found: {'max_depth': None, 'min